In [0]:
import joblib
import numpy as np
from pathlib import Path
from scipy.sparse import issparse

# Resolve /artifacts directory from this notebook's workspace path
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
nb_ws_path = ctx.notebookPath().get()                    # /Users/.../repo/notebooks/...
repo_ws_path = nb_ws_path.rsplit("/notebooks/", 1)[0]    # /Users/.../repo
ART = Path("/Workspace" + repo_ws_path) / "artifacts"

print("Artifacts path:", ART)

# Load pipeline (useful for feature names / explainability, even if you don't re-transform here)
pipeline = joblib.load(ART / "stedi_feature_pipeline.pkl")

# Load transformed features + labels
X_train_transformed = np.load(ART / "X_train_transformed.npy", allow_pickle=True)
X_test_transformed  = np.load(ART / "X_test_transformed.npy", allow_pickle=True)

y_train = joblib.load(ART / "y_train.pkl")
y_test  = joblib.load(ART / "y_test.pkl")

def to_float_matrix(arr: np.ndarray) -> np.ndarray:
    # Handle 0-d object holding a sparse matrix or array
    if getattr(arr, "ndim", None) == 0:
        arr = arr.item()
        if issparse(arr):
            arr = arr.toarray()
        return np.array(arr, dtype=float)

    # Object arrays containing per-row sparse matrices / arrays
    if getattr(arr, "dtype", None) == object:
        rows = []
        for x in arr:
            if issparse(x):
                rows.append(x.toarray())
            else:
                rows.append(np.array(x, dtype=float))
        return np.vstack(rows)

    # Sparse matrix
    if issparse(arr):
        return arr.toarray()

    # Normal numeric array
    return np.array(arr, dtype=float)

X_train = to_float_matrix(X_train_transformed)
X_test  = to_float_matrix(X_test_transformed)

y_train = np.ravel(y_train)
y_test  = np.ravel(y_test)

# Required sanity checks
print("X_train:", X_train.shape, X_train.dtype)
print("X_test :", X_test.shape, X_test.dtype)
print("y_train:", y_train.shape, np.unique(y_train, return_counts=True))
print("y_test :", y_test.shape,  np.unique(y_test,  return_counts=True))

assert X_train.shape[1] == X_test.shape[1]
assert X_train.shape[0] == y_train.shape[0]
assert X_test.shape[0] == y_test.shape[0]

print("NaNs X_train:", np.isnan(X_train).sum(), "Infs:", np.isinf(X_train).sum())
print("NaNs X_test :", np.isnan(X_test).sum(),  "Infs:", np.isinf(X_test).sum())


In [0]:
import numpy as np

def get_feature_names(pipeline, n_features: int):
    try:
        # many sklearn transformers support this if pipeline ends in something with get_feature_names_out
        names = pipeline.get_feature_names_out()
        names = [str(n) for n in names]
        if len(names) == n_features:
            return names
    except Exception:
        pass
    return [f"feature_{i}" for i in range(n_features)]

feature_names = get_feature_names(pipeline, X_train.shape[1])
len(feature_names), feature_names[:5]


In [0]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

baseline_lr = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("lr", LogisticRegression(
        solver="liblinear",
        penalty="l2",
        C=1.0,
        class_weight="balanced",
        max_iter=8000
    ))
])

baseline_lr.fit(X_train, y_train)

lr_model = baseline_lr.named_steps["lr"]
coefs = lr_model.coef_.ravel()

top = np.argsort(np.abs(coefs))[::-1][:15]
for i in top:
    print(feature_names[i], float(coefs[i]))


The largest Logistic Regression coefficients show that num__distance_cm is the strongest driver of predictions, and many of the next strongest features are one-hot columns for device_id and sensor_type. This is a concern because it suggests the model may be learning device-specific patterns rather than general step vs. no_step behavior, which can reduce reliability when the model is used on different devices or users. It also explains why performance is sensitive to class imbalance and thresholding: the features do not provide a clean separation, so changing the decision rule quickly shifts the balance between missed no_step cases and false positives.

In [0]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

pipe = Pipeline([
    ("scaler", StandardScaler(with_mean=False)),
    ("lr", LogisticRegression(max_iter=4000))
])

param_grid = {
    "lr__solver": ["liblinear"],
    "lr__penalty": ["l2"],
    "lr__C": [0.2, 0.5, 1, 2, 5],
    "lr__class_weight": ["balanced"],
}

search = GridSearchCV(
    pipe,
    param_grid=param_grid,
    scoring="balanced_accuracy",
    cv=cv,
    n_jobs=-1,
    verbose=2
)

search.fit(X_train, y_train)

best_lr = search.best_estimator_
print("Best params:", search.best_params_)
print("Best CV score:", search.best_score_)


In [0]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

def eval_model(model, X, y, title):
    pred = model.predict(X)
    print("\n", title)
    print("Balanced accuracy:", balanced_accuracy_score(y, pred))
    print("F1 macro:", f1_score(y, pred, average="macro"))
    print(classification_report(y, pred, zero_division=0))
    print("Confusion matrix [no_step, step]:")
    print(confusion_matrix(y, pred, labels=["no_step", "step"]))

eval_model(baseline_lr, X_test, y_test, "Old model (baseline LR)")
eval_model(best_lr, X_test, y_test, "New model (tuned LR)")


#Compare New Models
The new tuning did not improve performance. I will not switch models because the tuned model produced the exact same test predictions and confusion matrix as the baseline model.

In [0]:
old_pred = baseline_lr.predict(X_test)
new_pred = best_lr.predict(X_test)

old_bal = balanced_accuracy_score(y_test, old_pred)
new_bal = balanced_accuracy_score(y_test, new_pred)

print("Old test balanced accuracy:", old_bal)
print("New test balanced accuracy:", new_bal)

if new_bal > old_bal:
    joblib.dump(best_lr, str(ART / "stedi_best_model_lr_refined.pkl"))
    print("Saved refined model to:", ART / "stedi_best_model_lr_refined.pkl")
else:
    print("Kept old model; no save performed.")


# Model Refinement Summary

I ran a focused hyperparameter search for Logistic Regression using balanced accuracy because the dataset is imbalanced and accuracy alone can be misleading. I tuned the regularization strength (C) while keeping an L2 penalty and class_weight="balanced", and I compared solver settings, because these choices control overfitting and how strongly the model reacts to the minority class. The refined grid selected stronger regularization (C=0.2), but the tuned model did not improve performance on the test set. Both models had the same balanced accuracy (0.558), the same F1 macro (0.400), and the same confusion matrix, showing that these hyperparameter changes did not meaningfully shift the decision boundary. Because the tuned model was not better, I did not update the final saved model, which is responsible and ethical because it avoids claiming improvement that the evidence does not support.


# Ethics Reflection
Careless hyperparameter tuning can create unfair or unsafe models because it can optimize for a flattering metric like accuracy and still fail badly on rare but important cases, especially when the data is imbalanced. Examining model behavior carefully matters because the confusion matrix and class-level metrics reveal whether the model is actually detecting no_step or simply defaulting to the majority class. Explainability also helps by showing which features drive predictions, which can uncover overreliance on signals like device_id that may not generalize fairly across different devices or users. Gospel principles that guide my approach include honesty, integrity, and accountability, so I report results clearly and avoid overstating performance. Stewardship also applies here: I should make careful choices with data and models because the outputs can affect real people, and “by their fruits ye shall know them” reminds me to judge the model by its real outcomes, not just a single score.